In [ ]:
# @title
pip install numpy pandas matplotlib seaborn scikit-learn

## 📖 1. Краткий экскурс в теорию

### Постановка задачи
* **Классификация** — задача прогнозирования дискретного класса (например, «болен» / «здоров»).
* **Бинарная:** 2 класса (Положительный — наличие болезни, Отрицательный — норма).
* **Многоклассовая:** >2 классов (например, тип опухоли).

### Матрица ошибок и Метрики
| | Предсказано: Болен (1) | Предсказано: Здоров (0) |
|---|---|---|
| **Истинно: Болен (1)** | TP (True Positive) | FN (False Negative) |
| **Истинно: Здоров (0)** | FP (False Positive) | TN (True Negative) |

**Ключевые метрики:**
* **Accuracy (Точность):** $\frac{TP + TN}{TP + TN + FP + FN}$ — доля верных ответов. *Плохо работает при дисбалансе классов.*
* **Precision (Точность предсказания):** $\frac{TP}{TP + FP}$ — Из предсказанных "больных", сколько реально больны?
* **Recall (Полнота):** $\frac{TP}{TP + FN}$ — Из реальных "больных", сколько мы нашли? *Критично в медицине (лучше лишний раз отправить на перепроверку, чем пропустить болезнь).*
* **F1-мера:** $\frac{2 \cdot Precision \cdot Recall}{Precision + Recall}$ — гармоническое среднее.

In [ ]:
# @title
# Импорт библиотек и загрузка медицинских данных
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

# Загружаем датасет: рак молочной железы (569 образцов, 30 признаков)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target # 0 = malignant (злокачественная), 1 = benign (доброкачественная)

# Для удобства в медицине часто меняют местами, чтобы 1 означало "болезнь"
# В этом датасете 0 - это рак. Давайте инвертируем, чтобы 1 = Рак, 0 = Здоров
y = 1 - y

# Делим на тренировочную и тестовую выборки (80% на обучение, 20% на тест)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Размер обучающей выборки: {X_train.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")
print(f"Доля больных в тесте: {y_test.mean():.2%}")

## 🧠 2. Логистическая регрессия (LogReg)
Линейный классификатор. Находит разделяющую гиперплоскость и с помощью **сигмоиды** переводит расстояние до неё в вероятность $[0, 1]$.
Обучается путем минимизации **LogLoss (перекрестной энтропии)**.



### Задача
У нас есть медицинские данные пациентов. Нужно ответить на вопрос: **"Болен человек или здоров?"**
Это задача **бинарной классификации**: ответ всего два — 0 (здоров) или 1 (болен).

### Идея "на пальцах"
Представь, что у нас есть **один признак** — например, размер опухоли.
- Маленькая опухоль → скорее доброкачественная (близко к 0).
- Большая опухоль → скорее злокачественная (близко к 1).

Логистическая регрессия ищет **прямую линию**, которая лучше всего разделяет эти две группы.
Но есть проблема: линейная регрессия предсказывает числа от $-\infty$ до $+\infty$. А вероятность не может быть больше 1 или меньше 0!



## 🎯 Ключевая деталь — сигмоида

Обычная линейная функция даёт любое число: `-100, 0, 5, 42...`
Но вероятность должна быть от 0 до 1!

Поэтому мы "сжимаем" линейный ответ через **сигмоиду**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

- Если `z` большое положительное → сигмоида ≈ **1** ("точно болен")
- Если `z` большое отрицательное → сигмоида ≈ **0** ("точно здоров")
- Если `z = 0` → сигмоида = **0.5** ("не знаю, монетка")

👉 Сигмоида — это **мостик** между прямой линией и вероятностью.

## 🔄 Как линия превращается в вероятность — пошаговый переход

Представь этот процесс как **конвейер из двух шагов**.
Сначала мы считаем «сырое» расстояние, а потом «упаковываем» его в вероятность.

---

### 🏭 Конвейер предсказания

**Шаг 1. Линейная часть (считаем «расстояние»)**

Мы берём наши признаки ($x$), умножаем на веса ($w$) и прибавляем сдвиг ($b$):

$$z = w \cdot x + b$$

> **Результат:** любое число от $-\infty$ до $+\infty$. Это наша прямая линия.

**Шаг 2. Нелинейная часть (сжимаем в вероятность)**

Берём это число $z$ и «прогоняем» его через сигмоиду:

$$p = \sigma(z) = \frac{1}{1 + e^{-z}}$$

> **Результат:** число строго от 0 до 1. Это наша вероятность.

---

### 📊 Таблица-шпаргалка: что происходит с числами

Посмотри, как сигмоида «сплющивает» бесконечную линию в узкий коридор вероятностей:

| Сырой ответ линии ($z$) | Что это значит «на пальцах» | После сигмоиды ($p$) | Итоговый класс (при пороге 0.5) |
|:---:|:---|:---:|:---|
| $z = -10$ | Очень далеко от «больных» | 0.00004 | 🟢 Здоров (уверены на 99.9%) |
| $z = -2$ | Скорее здоров | 0.119 | 🟢 Здоров |
| $z = 0$ | Ровно на границе | 0.500 | 🟡 Монетка (50/50) |
| $z = 2$ | Скорее болен | 0.881 | 🔴 Болен |
| $z = 10$ | Очень далеко в сторону «больных» | 0.99995 | 🔴 Болен (уверены на 99.9%) |

---

### 🧠 Главная идея

- **Линейная часть** ($z = w \cdot x + b$) отвечает за **направление**: куда «смотрит» модель и как сильно признак сдвигает нас в сторону одного из классов.
- **Сигмоида** ($\sigma(z)$) отвечает за **форму**: она превращает любое число в корректную вероятность, не позволяя выйти за пределы $[0, 1]$.
- **Порог 0.5** — это «линия переключения»: слева от неё модель говорит «здоров», справа — «болен».

Почему такой порог? Можно ли поставить другой?

> 💡 Именно поэтому логистическая регрессия называется **логистической** — она использует логистическую (сигмоидную) функцию для перевода линейного ответа в вероятность.

In [ ]:
# @title 🔄 Как линия превращается в вероятность: два шага
import numpy as np
import matplotlib.pyplot as plt

# 1. Размер опухоли — только положительный (от 0 до 30 условных единиц)
x = np.linspace(0, 30, 200)

# 2. ШАГ 1: Линейная часть — «сырое» расстояние до разделяющей линии
# Подобраны такие веса: z = 0.8 * x - 12
# (чем больше опухоль → тем больше z → тем ближе к «болен»)
z = 0.8 * x - 12

# 3. ШАГ 2: Сигмоида — «сжимаем» z в вероятность от 0 до 1
p = 1 / (1 + np.exp(-z))

# --- Рисуем два графика друг под другом ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 9), sharex=True)

# ============ График 1: Линейная функция (ДО сигмоиды) ============
ax1.plot(x, z, color='blue', linewidth=2.5, label='z = 0.8·x − 12')
ax1.axhline(0, color='red', linestyle='--', linewidth=1.5,
            label='Граница (z = 0)')
ax1.fill_between(x, z, 0, where=(z > 0), color='red',   alpha=0.15)
ax1.fill_between(x, z, 0, where=(z < 0), color='green', alpha=0.15)

ax1.set_title("Шаг 1: Линейная функция — «сырой» ответ z", fontsize=14, fontweight='bold')
ax1.set_ylabel("Сырой ответ (z)", fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# ============ График 2: Сигмоида (ПОСЛЕ сжатия) ============
ax2.plot(x, p, color='green', linewidth=2.5, label='p = σ(z)')
ax2.axhline(0.5, color='red', linestyle='--', linewidth=1.5,
            label='Порог (p = 0.5)')
ax2.fill_between(x, p, 0.5, where=(p > 0.5), color='red',   alpha=0.15)
ax2.fill_between(x, p, 0.5, where=(p < 0.5), color='green', alpha=0.15)

# Найдём точку, где p = 0.5 (это x, где z = 0)
x_boundary = 12 / 0.8
ax2.axvline(x_boundary, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax2.annotate(f'Граница\nx ≈ {x_boundary:.1f}',
             xy=(x_boundary, 0.5),
             xytext=(x_boundary + 1.5, 0.25),
             arrowprops=dict(facecolor='gray', shrink=0.05),
             fontsize=10, color='dimgray',
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray"))

ax2.set_title("Шаг 2: Сигмоида — «сжали» в вероятность p ∈ [0, 1]",
              fontsize=14, fontweight='bold')
ax2.set_xlabel("Размер опухоли (x)", fontsize=11)
ax2.set_ylabel("Вероятность болезни (p)", fontsize=11)
ax2.set_ylim(-0.1, 1.1)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- Пояснение ---
print("🧠 Что произошло:")
print(f"• Прямая z уходит от {z.min():.0f} до {z.max():.0f} — это огромный диапазон.")
print(f"• Сигмоида сжала его в коридор [0, 1]: от {p.min():.4f} до {p.max():.4f}.")
print(f"• Точка переключения диагноза — там, где z = 0, т.е. при x ≈ {x_boundary:.1f}.")
print(f"• Меньше этого радиуса → модель говорит 'здоров', больше → 'болен'.")

## 🧩 Задание A. «Врач-аналитик»
Представьте, что вы — врач, которому принесли результаты логистической регрессии, обученной предсказывать злокачественность опухоли по её размеру.

На теоретическом графике ниже показана сигмоида: по оси X — размер опухоли, по оси Y — вероятность того, что опухоль злокачественная.

**Ответьте на вопросы:**

1. Пациент №1 пришёл с опухолью радиуса **10**. Модель говорит, что вероятность злокачественности ≈ **0.05**. Что вы скажете пациенту? Означает ли это, что он точно здоров?

2. Пациент №2 пришёл с опухолью радиуса **20**. Вероятность ≈ **0.88**. Как вы объясните ему результат, не используя слово «сигмоида»?

3. Есть ли такой размер опухоли, при котором модель «не уверена» — то есть вероятность около 0.5? Что в этой ситуации должен делать врач?

4. Почему для такой задачи **не подходит обычная линейная регрессия**? Приведите пример, что она могла бы выдать для очень большой опухоли, и почему это бессмысленно.

5. **Задача на подумать:** если мы поменяем порог с 0.5 на 0.3 (то есть будем считать «болен» уже при вероятности > 0.3) — что произойдёт с числом ложных тревог и пропущенных больных? В какую сторону это изменится?

# 💻 Задание «Допиши код»

Перед вами — почти готовый код. Ваша задача: заполнить пропуски

Не пугайтесь: **всё, что нужно, уже есть внутри библиотеки `sklearn`**. Мы просто берём готовые атрибуты модели и работаем с ними.

---

## 🔍 Как достать веса из модели — маленький пример

Прежде чем делать задание, посмотрим, как вообще устроена обученная модель.

Когда вы пишете `model.fit(X, y)`, внутри модели сохраняются **два числа**:

| Атрибут модели | Что хранит |
|---|---|
| `model.coef_` | веса $w$ для каждого признака |
| `model.intercept_` | сдвиг $b$ |

**Пример на игрушечных данных (просто чтобы посмотреть синтаксис):**


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
# Игрушка: 4 пациента, 1 признак
X_toy = np.array([[1.0], [2.0], [3.0], [4.0]]) #какие-то значения
y_toy = np.array([0, 0, 1, 1]) #здоров \ болен

toy_model = LogisticRegression(max_iter=10000)
toy_model.fit(X_toy, y_toy) #модель сама принимает (n_samples, n_features),


# Достаём веса:
# .coef_ это МАССИВ (потому что признаков может быть много),
# для одного признака берём [0][0]
w_toy = toy_model.coef_[0][0]
b_toy = toy_model.intercept_[0]

print(f"w = {w_toy:.3f}, b = {b_toy:.3f}")
print(f"Уравнение: z = {w_toy:.3f} * x + {b_toy:.3f}")

# Как получить вероятность? Через predict_proba:
probs = toy_model.predict_proba(X_toy)[:, 1]
print("Вероятности:", probs.round(3))

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# ============================================================
# ШАГ 1. ДАННЫЕ
# ============================================================
# Правило sklearn: X имеет форму (n_samples, n_features)
#   - строки  = объекты (пациенты)
#   - столбцы = признаки
#
# У нас: 4 пациента, 3 признака.
# Признаки назовём по-человечески:
#   - температура
#   - давление
#   - возраст
#
#                       температура  давление  возраст
X_toy = np.array([
    [36.6,        120.0,     25.0],   # пациент 0
    [36.7,        125.0,     30.0],   # пациент 1
    [38.5,        140.0,     55.0],   # пациент 2
    [39.0,        145.0,     60.0],   # пациент 3
])
# форма X_toy -> (4, 3): 4 пациента, 3 признака

# y — это ответ для каждого пациента: 0 = здоров, 1 = болен.
# Длина y должна совпадать с числом строк X (то есть с числом пациентов).
y_toy = np.array([0, 0, 1, 1])   # здоров, здоров, болен, болен


# ============================================================
# ШАГ 2. СОЗДАЁМ И ОБУЧАЕМ МОДЕЛЬ
# ============================================================
toy_model = LogisticRegression(max_iter=10000)

# fit сам понимает размеры:
#   - сколько строк = сколько пациентов (n_samples)
#   - сколько столбцов = сколько признаков (n_features)
#   - сколько разных значений в y = сколько классов
toy_model.fit(X_toy, y_toy)


# ============================================================
# ШАГ 3. ДОСТАЁМ ВЕСА
# ============================================================
# toy_model.coef_       — это МАССИВ формы (1, 3):
#                         [ [w1, w2, w3] ]
#                         первая ось = класс (бинарная задача -> 1 строка)
#                         вторая ось = признак (у нас 3 признака)
#
# toy_model.intercept_  — это МАССИВ формы (1,):
#                         [ b ]
#
# Поэтому берём [0] там, где нужна «строка номер 0»:
w = toy_model.coef_[0]          # -> [w1, w2, w3], три числа
b = toy_model.intercept_[0]     # -> b, одно число

w1, w2, w3 = w                  # распаковываем веса по признакам

print("Веса (по одному на каждый признак):")
print(f"  w1 (температура) = {w1:.3f}")
print(f"  w2 (давление)    = {w2:.3f}")
print(f"  w3 (возраст)     = {w3:.3f}")
print(f"  b  (сдвиг)       = {b:.3f}")
print()

print("Уравнение модели:")
print(f"  z = {w1:.3f} * температура + {w2:.3f} * давление + {w3:.3f} * возраст + ({b:.3f})")
print(f"  p = 1 / (1 + exp(-z))       # вероятность быть больным")
print(f"  класс = 1, если p >= 0.5, иначе 0")
print()


# ============================================================
# ШАГ 4. СЧИТАЕМ ВЕРОЯТНОСТИ
# ============================================================
# predict_proba возвращает массив формы (n_samples, n_classes).
# Каждая строка = [P(класс 0), P(класс 1)] для одного пациента.
# Брать столбец [:, 1] — значит взять вероятность класса 1 (болен).
probs = toy_model.predict_proba(X_toy)[:, 1]

print("Вероятность быть больным для каждого пациента:")
for i, p in enumerate(probs):
    print(f"  пациент {i}: x = {X_toy[i]},  p(болен) = {p:.3f}")


# ============================================================
# ШАГ 5. ПРОВЕРЯЕМ ФОРМУЛУ РУКАМИ
# ============================================================
# Возьмём первого пациента и вручную посчитаем то же самое,
# что делает модель внутри. Результат должен совпасть с probs[0].
x_new = X_toy[0]                 # [36.6, 120.0, 25.0]

z = w1 * x_new[0] + w2 * x_new[1] + w3 * x_new[2] + b
p_manual = 1.0 / (1.0 + np.exp(-z))

print()
print("Ручная проверка для пациента 0:")
print(f"  z = {z:.4f}")
print(f"  p = 1 / (1 + exp(-z)) = {p_manual:.4f}")
print(f"  для сравнения predict_proba дал: {probs[0]:.4f}")

Представьте, что вы — data scientist в кардиологической клинике. К вам приходят пациенты, и нужно понять: есть ли у них риск сердечного приступа.

In [ ]:
# @title Задание C: Сердечный приступ 🫀

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ===========================================================
# 📦 ШАГ 1. Генерируем синтетические данные пациентов
# ===========================================================
# Это уже сделано за тебя — просто запускай ячейку

np.random.seed(42)
n_patients = 300

# Генерируем признаки
age = np.random.randint(30, 70, n_patients)           # возраст 30-70 лет
cholesterol = np.random.randint(150, 300, n_patients) # холестерин 150-300 мг/дл
bp = np.random.randint(100, 180, n_patients)          # давление 100-180 мм рт.ст.

# Генерируем целевую переменную (риск)
# Чем старше, выше холестерин и давление — тем выше риск
risk_score = 0.05 * age + 0.02 * cholesterol + 0.03 * bp - 10
probability = 1 / (1 + np.exp(-risk_score))
risk = (probability > 0.5).astype(int)

# Собираем в таблицу
df = pd.DataFrame({
    'age': age,
    'cholesterol': cholesterol,
    'bp': bp,
    'risk': risk
})

print(f"📊 Данные по {n_patients} пациентам:")
print(df.head(10))
print(f"\n📈 Статистика:")
print(f"   Пациентов с риском: {risk.sum()} ({risk.mean()*100:.1f}%)")
print(f"   Пациентов без риска: {n_patients - risk.sum()} ({(1-risk.mean())*100:.1f}%)")


#Как эти три строки связаны с лекцией и семинарами ранее?
risk_score = 0.05 * age + 0.02 * cholesterol + 0.03 * bp - 10

probability = 1 / (1 + np.exp(-risk_score))  

risk = (probability > 0.5).astype(int)    

Напоминаю про код выше:


In [ ]:
# Собираем в таблицу
df = pd.DataFrame({
    'age': age,
    'cholesterol': cholesterol,
    'bp': bp,
    'risk': risk
})

Задача: в пункты X и Y правильно впишите

In [ ]:
# ===========================================================
# 🎓 ШАГ 2. Готовим данные для модели
# ===========================================================
# Модели нужны признаки (X) и целевая переменная (y)
# X — это таблица без колонки 'risk'
# y — это колонка 'risk'

X = df[['*', '*', '*']]  # признаки
y = df['*']                        # целевая переменная

# Важно! Признаки нужно нормализовать (привести к одному масштабу)
# Иначе модель будет считать, что холестерин (150-300) важнее возраста (30-70)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\n✅ Данные готовы!")
print(f"   X shape: {X_scaled.shape} (300 пациентов × 3 признака)")
print(f"   y shape: {y.shape} (300 меток)")

In [ ]:
# ===========================================================
# 🔍 TODO 1: Обучи модель
# ===========================================================
# Создай модель логистической регрессии и обучи её
# Подсказка, смотри как выше было:
#   model = ****(max_iter=1000)
#   model.fit(*, *)

#model = ...   # ← допиши: создай модель (max_iter=1000)
#...           # ← допиши: обучи модель на X_scaled и

model = ****(max_iter=1000)
model.fit(*, *)

print(f"\n✅ Модель обучена!")


In [ ]:
# ===========================================================
# 📊 TODO 2: Посмотри, какие признаки важнее
# ===========================================================
# У модели есть атрибут model.coef_ — это веса признаков
# Чем больше вес (по модулю), тем важнее признак

weights = model.coef_[0]  # веса для трёх признаков

print(f"\n🔢 Веса модели (важность признаков):")
print(f"   Возраст:       w1 = {weights[0]:+.3f}")
print(f"   Холестерин:    w2 = {weights[1]:+.3f}")
print(f"   Давление:      w3 = {weights[2]:+.3f}")

# Визуализируем важность признаков
plt.figure(figsize=(8, 4))
plt.bar(['Возраст', 'Холестерин', 'Давление'], np.abs(weights), color=['blue', 'orange', 'green'])
plt.ylabel('Важность (|w|)')
plt.title('Какие признаки важнее для предсказания риска?')
plt.grid(True, alpha=0.3)
plt.show()

# 📏 3. Метод k-ближайших соседей (k-NN) и Масштабирование

k-NN — «ленивый» алгоритм: он не строит явную модель, а просто запоминает все объекты
и на этапе предсказания ищет ближайшие к новому объекту.

Чтобы найти «ближайших», нужно вычислить **расстояние**. Чаще всего используют
евклидово расстояние:

$$d(x, y) = \sqrt{\sum_{j=1}^{n} (x_j - y_j)^2}$$

⚠️ **Важно:** Если признаки имеют разный масштаб (например, уровень глюкозы = 7.0,
а полигенный риск-скор = 0.1), признак с большими числами «перетянет» евклидово
расстояние на себя!

**Решение:** Всегда делаем `StandardScaler` перед k-NN.

Если по простому:

k-NN — это врач, который не учил теорию, а просто помнит всех прошлых пациентов. Когда приходит новый (Query), он говорит:

«Хм, этот пациент по своим анализам больше всего похож на Петю, у которого был диабет. Значит, и у этого, скорее всего, диабет.»

## 🍳 Аналогия: два студента перед экзаменом

**Студент A («прилежный»)** — это, например, логистическая регрессия или решающее дерево:

- За месяц до экзамена сел, прочитал все лекции, составил конспект, вывел формулы.
- К экзамену пришёл с **готовой моделью** в голове: *«Если глюкоза > 6.5 и риск-скор > 0.5 → диабет»*.
- На экзамене ему достаточно **подставить числа в формулу** — ответ готов мгновенно.

**Студент B («ленивый»)** — это k-NN:

- Ничего не учил. Просто **принёс с собой все лекции и учебники** (всю обучающую выборку).
- На экзамене ему задают вопрос про нового пациента. Он такой: *«Так, сейчас найду похожих...»*
- Начинает **листать все лекции**, сравнивать нового пациента с каждым из старых, искать самого похожего.
- Только после этого отвечает.

Вот почему k-NN «ленивый»: он **откладывает всю работу на момент предсказания**.



### 🧬 Пример: диагностика диабета 2 типа
Данные трёх известных пациентов и одного нового (Query):

| Пациент | Глюкоза натощак (ммоль/л) | Полигенный риск-скор | Диагноз |
|---------|---------------------------|----------------------|---------|
| P1      | 7.0                       | 0.10                 | Диабет  |
| P2      | 5.0                       | 0.85                 | Диабет  |
| P3      | 6.5                       | 0.20                 | Норма   |
| **Query**| **6.0**                   | **0.80**             | **?**   |

Попробуем классифицировать Query при $k=1$ **без масштабирования**:
* Расстояние до P2 (Диабет): $\sqrt{(6.0 - 5.0)^2 + (0.80 - 0.85)^2} = \sqrt{1.0 + 0.0025} \approx \mathbf{1.001}$
* Расстояние до P3 (Норма): $\sqrt{(6.0 - 6.5)^2 + (0.80 - 0.20)^2} = \sqrt{0.25 + 0.36} \approx \mathbf{0.781}$

**Результат:** $0.781 < 1.001$, поэтому ближайший сосед — P3. Модель предскажет **«Норма»**.
Это опасная ошибка (False Negative), возникшая только потому, что разница в глюкозе (1.0) математиче

## 📊 Визуализация ловушки k-NN без масштабирования

Ниже мы построим два графика.
- **Слева:** Исходные данные. Из-за того, что шкала глюкозы (4–8) шире шкалы риска (0–1), алгоритм математически считает точку P3 ближе к Query, хотя интуитивно Query гораздо больше похож на P2 (высокий риск).
- **Справа:** Данные после `StandardScaler`. Теперь оба признака равноправны (среднее = 0, отклонение = 1), и алгоритм честно находит истинного ближайшего соседа — P2.

In [ ]:
# @title
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Исходные данные
X_orig = np.array([[7.0, 0.10],   # P1 (Диабет)
                   [5.0, 0.85],   # P2 (Диабет)
                   [6.5, 0.20],   # P3 (Норма)
                   [6.0, 0.80]])  # Query (?)
y_orig = ['P1', 'P2', 'P3', 'Query']
labels = ['P1 (Диабет)', 'P2 (Диабет)', 'P3 (Норма)', 'Query (?)']

# 2. Масштабированные данные
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_orig)

def plot_knn_space(X, y, labels, title, ax, is_scaled=False):
    colors = ['red', 'red', 'blue', 'gold']
    markers = ['o', 'o', 'o', '*']

    # Рисуем точки
    for i in range(len(X)):
        size = 250 if i == 3 else 120
        ax.scatter(X[i, 0], X[i, 1], c=colors[i], marker=markers[i],
                   s=size, edgecolor='black', linewidth=2, zorder=5)

        # Подписи точек
        if i == 0:  # P1 - вправо и вниз
            offset = (0.08, -0.08)
        elif i == 1:  # P2 - влево и вверх
            offset = (-0.25, 0.08)
        elif i == 2:  # P3 - вправо
            offset = (0.08, 0.05)
        else:  # Query - вверх
            offset = (0.05, 0.12)

        ax.text(X[i, 0] + offset[0], X[i, 1] + offset[1], labels[i],
                fontsize=10, fontweight='bold', zorder=6)

    query_point = X[3]
    p2_point = X[1]
    p3_point = X[2]

    # Считаем расстояния
    dist_p2 = np.linalg.norm(query_point - p2_point)
    dist_p3 = np.linalg.norm(query_point - p3_point)

    # Рисуем ЛИНИИ только к P2 и P3 (P1 не участвует в сравнении)
    ax.plot([query_point[0], p2_point[0]], [query_point[1], p2_point[1]],
            'r--', alpha=0.7, linewidth=2.5, zorder=3, label=f'До P2: {dist_p2:.2f}')
    ax.plot([query_point[0], p3_point[0]], [query_point[1], p3_point[1]],
            'b--', alpha=0.7, linewidth=2.5, zorder=3, label=f'До P3: {dist_p3:.2f}')

    # Подписи расстояний в белых бейджиках
    props = dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray',
                 alpha=0.95, linewidth=1.5)

    mid_p2 = (query_point + p2_point) / 2
    mid_p3 = (query_point + p3_point) / 2

    # Позиционируем подписи так, чтобы не перекрывать точки
    ax.text(mid_p2[0] - 0.15, mid_p2[1] + 0.1, f'd = {dist_p2:.2f}',
            fontsize=11, color='red', fontweight='bold', bbox=props, zorder=7)
    ax.text(mid_p3[0] + 0.05, mid_p3[1] - 0.15, f'd = {dist_p3:.2f}',
            fontsize=11, color='blue', fontweight='bold', bbox=props, zorder=7)

    # Определяем победителя
    if dist_p2 < dist_p3:
        winner_text = "P2 (Диабет) - ВЕРНО"
        winner_color = 'green'
    else:
        winner_text = "P3 (Норма) - ОШИБКА"
        winner_color = 'darkred'

    ax.set_title(f"{title}\nБлижайший сосед: {winner_text}",
                 fontsize=13, fontweight='bold', color=winner_color, pad=15)

    # Подписи осей
    if is_scaled:
        ax.set_xlabel('Глюкоза (стандартизированная)', fontsize=11)
        ax.set_ylabel('Риск-скор (стандартизированный)', fontsize=11)
    else:
        ax.set_xlabel('Глюкоза натощак (ммоль/л)', fontsize=11)
        ax.set_ylabel('Полигенный риск-скор', fontsize=11)

    ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
    ax.grid(True, linestyle=':', alpha=0.6)

    # Устанавливаем одинаковые пропорции осей для корректного отображения расстояний
    ax.set_aspect('equal', adjustable='box')

# 3. Отрисовка
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

plot_knn_space(X_orig, y_orig, labels, "БЕЗ масштабирования", ax1, is_scaled=False)
plot_knn_space(X_scaled, y_orig, labels, "С масштабированием (StandardScaler)", ax2, is_scaled=True)

plt.tight_layout()
plt.show()

## 🌈 4. Многоклассовая классификация: OvR vs Softmax

Когда классов больше двух (например, 3 типа опухолей: A, B, C), линейные модели решают задачу двумя способами:

### 🔹 One-vs-Rest (OvR) — «Один против всех»
Для **каждого класса** обучается своя бинарная модель:
- Модель 1: класс A vs (B + C)
- Модель 2: класс B vs (A + C)
- Модель 3: класс C vs (A + B)

На выходе — 3 независимые вероятности. Класс выбирается по максимальной.

### 🔹 Softmax (Multinomial) — «Сразу все классы»
Модель обучается **сразу на все классы** и выдаёт вектор логитов, который через функцию Softmax превращается в распределение вероятностей:

$$P(y=i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}, \quad \sum_{i=1}^{K} P(y=i) = 1$$

Сумма вероятностей **всегда равна 1** — это ключевое отличие от OvR.\

 Ниже мы визуализируем оба подхода на данных с 3 классами (например, 3 типа клеток).

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Генерируем 3 четких кластера (имитация 3 типов клеток)
X, y = make_blobs(n_samples=300, centers=3, cluster_std=1.0,
                  random_state=42, n_features=2)
scaler = StandardScaler()
X = scaler.fit_transform(X)

class_names = ['Тип A (норма)', 'Тип B (доброкач.)', 'Тип C (злокач.)']
colors = ['#2ecc71', '#f39c12', '#e74c3c']  # зеленый, оранжевый, красный

# 2. Обучаем две модели
model_ovr = LogisticRegression(multi_class='ovr', solver='lbfgs', random_state=42)
model_soft = LogisticRegression(multi_class='multinomial', solver='lbfgs', random_state=42)
model_ovr.fit(X, y)
model_soft.fit(X, y)

# 3. Функция для визуализации границ решений
def plot_decision_boundary(model, X, y, title, ax, show_probs=False):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    # Получаем вероятности для каждой точки сетки
    Z_probs = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])
    Z_class = Z_probs.argmax(axis=1).reshape(xx.shape)

    # Рисуем фон по классам
    ax.contourf(xx, yy, Z_class, alpha=0.25, cmap='RdYlGn')

    # Рисуем точки
    for i, color in enumerate(colors):
        mask = y == i
        ax.scatter(X[mask, 0], X[mask, 1], c=color, edgecolor='black',
                   s=40, label=class_names[i], zorder=5)

    # Если OvR — рисуем все три разделяющие линии (где вероятность = 0.5)
    if hasattr(model, 'coef_') and model.coef_.shape[0] == 3:
        for i in range(3):
            w = model.coef_[i]
            b = model.intercept_[i]
            x_line = np.array([x_min, x_max])
            y_line = -(w[0] * x_line + b) / w[1]
            ax.plot(x_line, y_line, '--', color=colors[i], linewidth=2,
                    label=f'Граница класса {chr(65+i)}', alpha=0.7)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Признак 1')
    ax.set_ylabel('Признак 2')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.5)

# 4. Рисуем OvR: три подграфика — каждый показывает свою бинарную границу
fig = plt.figure(figsize=(16, 6))

# Левая часть: 3 подграфика OvR
for i in range(3):
    ax = fig.add_subplot(1, 4, i + 1)
    # Создаем бинарную задачу: класс i vs остальные
    y_binary = (y == i).astype(int)
    model_bin = LogisticRegression(random_state=42).fit(X, y_binary)

    # Рисуем точки: выделенный класс — цветным, остальные — серым
    ax.scatter(X[y == i, 0], X[y == i, 1], c=colors[i],
               edgecolor='black', s=60, label=f'Класс {chr(65+i)}', zorder=5)
    ax.scatter(X[y != i, 0], X[y != i, 1], c='lightgray',
               edgecolor='gray', s=30, label='Остальные', zorder=4)

    # Рисуем разделяющую линию
    w = model_bin.coef_[0]
    b = model_bin.intercept_[0]
    x_line = np.array([X[:, 0].min() - 0.5, X[:, 0].max() + 0.5])
    y_line = -(w[0] * x_line + b) / w[1]
    ax.plot(x_line, y_line, '--', color=colors[i], linewidth=2.5)

    ax.set_title(f'OvR: Модель {i+1}\n"{class_names[i]}" vs остальные',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Признак 1')
    ax.set_ylabel('Признак 2')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.5)

# Правая часть: Softmax — одна общая граница
ax_soft = fig.add_subplot(1, 4, 4)
plot_decision_boundary(model_soft, X, y, 'Softmax (Multinomial)\nОдна общая граница', ax_soft)

plt.tight_layout()
plt.show()

Именно multi_class='multinomial' говорит scikit-learn использовать многоклассовую логистическую регрессию, которая внутри применяет Softmax.
solver='lbfgs' — это оптимизатор, который поддерживает такой режим.

### 🎯 Что видно на визуализации:

**Слева (OvR — три отдельных модели):**
- Каждая из 3 моделей строит **свою** разделяющую линию (пунктир того же цвета, что и класс)
- Модель 1 отделяет зеленый кластер (A) от всех остальных
- Модель 2 — оранжевый (B), Модель 3 — красный (C)
- ⚠️ Проблема: в зонах пересечения линий могут возникать **конфликты** (несколько моделей одновременно предсказывают "свой" класс) или **пустоты** (ни одна не уверена)

**Справа (Softmax — одна модель):**
- Все три класса рассматриваются **одновременно**
- Пространство делится на 3 чёткие зоны без конфликтов
- Сумма вероятностей для любой точки = 1

###  Сравнение на примере одного пациента:

| Подход | Вероятность A | Вероятность B | Вероятность C | Сумма |
|--------|--------------|--------------|--------------|-------|
| **OvR** | 0.15 | 0.72 | 0.20 | **1.07** ❌ |
| **Softmax** | 0.12 | 0.75 | 0.13 | **1.00** ✅ |

В OvR вероятности **независимы** и могут не давать в сумме 1. В Softmax — это настоящее распределение вероятностей, что критично для медицинской интерпретации ("у пациента 75% вероятности типа B").

## 🎯 5. Матрица ошибок и рождение метрик

Всё начинается с **Матрицы ошибок (Confusion Matrix)**. Это таблица, которая сопоставляет истинные диагнозы с предсказаниями модели.

В медицине «Положительный класс» (1) — это обычно наличие заболевания, которое мы ищем.

| | Предсказано: Здоров (0) | Предсказано: Болен (1) |
| :--- | :---: | :---: |
| **Истинно: Здоров (0)** | **TN** (True Negative)<br>Верно: здоров | **FP** (False Positive)<br>Ложная тревога |
| **Истинно: Болен (1)** | **FN** (False Negative)<br>Пропуск болезни | **TP** (True Positive)<br>Верно: болен |

Из этих 4-х чисел «рождаются» все основные метрики качества.

In [ ]:
# @title
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 1. Создаем "идеальные" данные для наглядного счета
# 0 = Здоров, 1 = Болен
y_true = np.array([0, 0, 0, 0, 0,  1, 1, 1, 1, 1]) # 5 здоровых, 5 больных
y_pred = np.array([0, 0, 0, 1, 1,  1, 1, 1, 0, 0]) # Предсказания модели

# 2. Строим матрицу ошибок
# sklearn возвращает матрицу в формате: [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("=== Матрица ошибок ===")
print(f"TN (Верно здоров): {tn}")
print(f"FP (Ложная тревога): {fp}")
print(f"FN (Пропуск болезни): {fn}")
print(f"TP (Верно болен): {tp}")
print("-" * 30)

# 3. РОЖДЕНИЕ МЕТРИК: считаем вручную по формулам
accuracy_manual = (tp + tn) / (tp + tn + fp + fn)
precision_manual = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_manual = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual) if (precision_manual + recall_manual) > 0 else 0

print("=== Метрики (ручной расчет) ===")
print(f"Accuracy  (Доля верных):     {accuracy_manual:.2f}  = ({tp}+{tn}) / 10")
print(f"Precision (Точность):        {precision_manual:.2f}  = {tp} / ({tp}+{fp})")
print(f"Recall    (Полнота):         {recall_manual:.2f}  = {tp} / ({tp}+{fn})")
print(f"F1-мера   (Баланс):          {f1_manual:.2f}")

print("\n=== Проверка через sklearn ===")
print(f"Accuracy:  {accuracy_score(y_true, y_pred):.2f}")
print(f"Precision: {precision_score(y_true, y_pred):.2f}")
print(f"Recall:    {recall_score(y_true, y_pred):.2f}")
print(f"F1:        {f1_score(y_true, y_pred):.2f}")

### 🩺 Медицинская интерпретация метрик

Почему нам недостаточно просто Accuracy? Потому что цена ошибок разная!

1. **Accuracy (Точность):** «Какой процент всех пациентов мы классифицировали верно?»
   - *Плохо для медицины:* Если болезнь редкая (1% населения), модель, которая всем ставит «Здоров», будет иметь Accuracy 99%, но будет абсолютно бесполезной.

2. **Precision (Точность предсказания):** «Если модель сказала "Болен", насколько ей можно верить?»
   - *Цена FP (Ложной тревоги):* Лишний стресс пациента, направление на дорогие и инвазивные дообследования (например, биопсию).

3. **Recall (Полнота / Чувствительность):** «Какую долю реально больных людей мы смогли найти?»
   - *Цена FN (Пропуска):* Пациента отправили домой, болезнь прогрессирует. **В скрининге опасных заболеваний (рак, ВИЧ) Recall — самая важная метрика.**

4. **F1-мера:** Гармоническое среднее между Precision и Recall. Используется, когда нужно найти баланс (например, классы несбалансированы).

In [ ]:
# @title
import numpy as np
from sklearn.metrics import confusion_matrix

# 1. Сценарий: Редкое заболевание. Из 1000 пациентов болеют только 10 (1%).
# Модель-пессимист: всегда предсказывает "Здоров" (0), чтобы просто не ошибаться.
y_true_rare = np.array([0]*990 + [1]*10)
y_pred_dumb = np.array([0]*1000)

# 2. Смотрим, что попало в матрицу ошибок
cm = confusion_matrix(y_true_rare, y_pred_dumb)
tn, fp, fn, tp = cm.ravel()

print("=== 🚨 ЛОВУШКА ACCURACY: Редкая болезнь (1%) 🚨 ===")
print(f"Всего пациентов: 1000 (990 здоровых, 10 больных)")
print(f"Предсказания модели: ВСЕГДА 'Здоров' (0)")
print("-" * 55)
print("📊 Матрица ошибок этой 'глупой' модели:")
print(f"  TN (Верно здоров): {tn}")
print(f"  FP (Ложная тревога): {fp}")
print(f"  FN (Пропуск болезни): {fn}")
print(f"  TP (Верно болен): {tp}")
print("-" * 55)

# 3. РОЖДЕНИЕ МЕТРИК: считаем вручную, подставляя эти числа в формулы
accuracy_manual = (tp + tn) / (tp + tn + fp + fn)
precision_manual = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall_manual = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual) if (precision_manual + recall_manual) > 0 else 0.0

print("🧮 Ручной расчет метрик (подстановка в формулы):")
print(f"  Accuracy  = (TP + TN) / Всего = ({tp} + {tn}) / 1000 = {accuracy_manual:.1%}  <-- Выглядит потрясающе! 🎉")
print(f"  Precision = TP / (TP + FP)    = {tp} / ({tp} + {fp}) = {precision_manual:.1%}  <-- Деление на ноль (модель никого не назвала больным)")
print(f"  Recall    = TP / (TP + FN)    = {tp} / ({tp} + {fn}) = {recall_manual:.1%}  <-- На самом деле модель не нашла НИ ОДНОГО больного 💀")
print(f"  F1-мера   = 2 * (P * R) / (P + R) = {f1_manual:.1%}")
print("-" * 55)
print("💡 ГЛАВНЫЙ ВЫВОД СЕМИНАРА:")
print("Никогда не смотрите только на Accuracy в задачах с дисбалансом классов!")
print("В медицине всегда проверяйте Recall (чтобы не пропустить болезнь) и F1-меру.")

## 📉 6. Функция потерь для классификации: Перекрёстная энтропия (Log Loss)

Когда мы обучаем модель (например, логистическую регрессию), ей нужно понять, **насколько она ошибается**, чтобы скорректировать свои веса.

Для регрессии используют MSE (среднеквадратичную ошибку). Но для классификации, где на выходе **вероятность** $p \in [0, 1]$, используют **Перекрёстную энтропию (Cross-Entropy)** или **Log Loss**.

### Формула для бинарной классификации:
$$Loss = - \left[ y \cdot \log(p) + (1 - y) \cdot \log(1 - p) \right]$$

Где:
* $y$ — истинная метка (0 или 1)
* $p$ — предсказанная моделью вероятность класса 1

### Как это работает (два случая):
1. **Если пациент болен ($y = 1$):** Формула превращается в $Loss = -\log(p)$.
   Чем ближе $p$ к 1, тем меньше штраф (близко к 0). Чем ближе $p$ к 0, тем штраф **стремится к бесконечности**.
2. **Если пациент здоров ($y = 0$):** Формула превращается в $Loss = -\log(1 - p)$.
   Чем ближе $p$ к 0, тем меньше штраф. Чем ближе $p$ к 1, тем штраф огромный.

> 💡 **Простая аналогия:** Это «мера удивления». Если модель уверена ($p=0.99$), а случается обратное ($y=0$), модель «шокирована», и штраф становится огромным.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Генерируем вероятности от 0.01 до 0.99 (избегаем log(0))
p = np.linspace(0.01, 0.99, 100)

# Считаем Loss для двух случаев
loss_y1 = -np.log(p)       # Истинный класс y = 1 (Болен)
loss_y0 = -np.log(1 - p)   # Истинный класс y = 0 (Здоров)

plt.figure(figsize=(10, 6))

# График для y = 1 (Синий)
plt.plot(p, loss_y1, label='Истинный класс: y = 1 (Болен)', color='blue', linewidth=3)
# График для y = 0 (Красный)
plt.plot(p, loss_y0, label='Истинный класс: y = 0 (Здоров)', color='red', linewidth=3)

# Аннотации для наглядности
plt.annotate('Штраф → ∞\n(Модель уверена, что здоров,\nа пациент болен)',
             xy=(0.05, 3), xytext=(0.2, 4),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5),
             fontsize=10, color='blue', fontweight='bold')

plt.annotate('Штраф → ∞\n(Модель уверена, что болен,\nа пациент здоров)',
             xy=(0.95, 3), xytext=(0.6, 4),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5),
             fontsize=10, color='red', fontweight='bold')

plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
plt.text(0.51, 0.1, 'p = 0.5\n(Неопределенность)', fontsize=9, color='gray')

plt.title('Перекрёстная энтропия (Log Loss): Штраф за уверенность в ошибке', fontsize=14, fontweight='bold')
plt.xlabel('Предсказанная вероятность класса 1 (p)', fontsize=12)
plt.ylabel('Значение функции потерь (Loss)', fontsize=12)
plt.xlim(0, 1)
plt.ylim(0, 5) # Ограничиваем Y, чтобы график был читаемым
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

### 🩺 Медицинская интерпретация графика

Посмотрите на синюю линию ($y=1$, пациент реально болен):
* Если модель предсказывает $p = 0.9$ (уверена, что болен), $Loss \approx 0.1$. **Штраф маленький, модель молодец.**
* Если модель предсказывает $p = 0.5$ (не знает), $Loss \approx 0.69$. **Штраф средний.**
* Если модель предсказывает $p = 0.01$ (уверена, что здоров!), $Loss \approx 4.6$. **Катастрофический штраф!**

Именно этот «бесконечный» штраф заставляет алгоритм градиентного спуска активно менять веса модели, чтобы избежать таких фатальных ошибок в будущем.

### Почему не использовать обычную ошибку (MSE)?
Если попытаться обучить логистическую регрессию через MSE, функция потерь станет «волнистой» (невыпуклой). Алгоритм оптимизации может застрять в локальном минимуме и никогда не найти лучшие веса. Перекрёстная энтропия гарантирует наличие **одного глобального минимума**, что делает обучение стабильным и предсказуемым.